# Step 10 - Statistical Comparison of Models

Test whether performance differences are statistically significant: McNemar's test (pairwise, same test set), the Friedman omnibus test (across models over CV folds), and the Nemenyi post-hoc with a critical difference.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
import joblib
from src import data, models, benchmark, stats_tests as st
from src import utils
from sklearn.metrics import f1_score

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
store_path = C.MODELS_DIR / "test_predictions.joblib"
if store_path.exists():
    store = joblib.load(store_path)
    predictions = store["predictions"]
else:
    zoo = models.get_model_zoo(x_train)
    _, _, predictions = benchmark.evaluate_on_test(x_train, y_train, x_test, y_test, zoo)
NAMES = list(predictions.keys())
print("Models:", NAMES)

Models: ['Logistic Regression', 'Ridge Classifier', 'SGD Classifier', 'Decision Tree', 'Random Forest', 'Extra Trees', 'AdaBoost', 'Gradient Boosting', 'KNN', 'GaussianNB', 'SVM', 'MLP', 'XGBoost', 'LightGBM', 'CatBoost']


## 8.1 McNemar pairwise tests

In [3]:
mc = st.mcnemar_pairwise(y_test.values, predictions, NAMES)
display(mc.head(25))
utils.save_table(mc, "mcnemar_pairwise",
                 caption="McNemar pairwise tests on the held-out set.", label="tab:mcnemar")
n_sig = int(mc["significant_0.05"].sum())
print(f"Significant pairs (p<0.05): {n_sig} / {len(mc)}")

,Model A,Model B,"b (A right, B wrong)","c (A wrong, B right)",statistic,p_value,significant_0.05
0,GaussianNB,XGBoost,2,332,324.0749,1.876411e-72,True
1,GaussianNB,CatBoost,2,332,324.0749,1.876411e-72,True
2,Extra Trees,GaussianNB,332,2,324.0749,1.876411e-72,True
3,Decision Tree,GaussianNB,331,2,323.0751,3.098084e-72,True
4,Random Forest,GaussianNB,330,2,322.0753,5.115172e-72,True
5,GaussianNB,LightGBM,2,330,322.0753,5.115172e-72,True
6,Gradient Boosting,GaussianNB,328,2,320.0758,1.394439e-71,True
7,KNN,GaussianNB,329,3,318.1476,3.667750e-71,True
8,GaussianNB,MLP,2,326,318.0762,3.801423e-71,True
9,AdaBoost,GaussianNB,324,4,310.2470,1.929426e-69,True


Significant pairs (p<0.05): 64 / 105


## 8.2 Friedman omnibus test over CV folds

In [4]:
zoo = models.get_model_zoo(x_train)
score_matrix = st.cv_score_matrix(x_train, y_train, zoo,
                                  lambda yt, yp: f1_score(yt, yp, zero_division=0))
display(score_matrix.round(4))
fried = st.friedman_test(score_matrix)
print("Friedman:", fried)
utils.save_table(score_matrix.round(6), "cv_score_matrix")
utils.save_json(fried, "friedman_test")

,Logistic Regression,Ridge Classifier,SGD Classifier,Decision Tree,Random Forest,Extra Trees,AdaBoost,Gradient Boosting,KNN,GaussianNB,SVM,MLP,XGBoost,LightGBM,CatBoost
0,0.8900,0.7969,0.8922,0.9965,0.9982,0.9982,0.9842,0.9991,0.9965,0.6953,0.9835,0.9965,0.9982,0.9974,0.9982
1,0.8940,0.8149,0.8993,0.9938,0.9974,0.9974,0.9823,0.9965,0.9938,0.7179,0.9844,0.9947,0.9965,0.9965,0.9965
2,0.9080,0.7976,0.9014,0.9974,0.9982,0.9982,0.9816,0.9921,0.9939,0.6995,0.9844,0.9956,0.9965,0.9965,0.9974
3,0.9066,0.8285,0.9057,0.9974,1.0000,1.0000,0.9816,0.9991,0.9956,0.7924,0.9878,0.9974,0.9991,0.9982,1.0000
4,0.9042,0.8138,0.8983,0.9973,0.9991,0.9982,0.9807,0.9974,0.9965,0.7562,0.9826,0.9956,0.9991,0.9973,0.9991


Friedman: {'statistic': 65.56085229324664, 'p_value': 1.2146839507144438e-08, 'significant_0.05': True, 'n_models': 15, 'n_folds': 5}


PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/friedman_test.json')

## 8.3 Nemenyi post-hoc and critical-difference ranking

In [5]:
nem = st.nemenyi_posthoc(score_matrix)
print("Critical difference (alpha=0.05):", nem["critical_difference"])
ranks = pd.Series(nem["avg_ranks"]).sort_values()
display(ranks.to_frame("avg_rank"))
utils.save_table(ranks.reset_index().rename(columns={"index":"Model",0:"avg_rank"}),
                 "nemenyi_ranks", caption="Average Friedman ranks (1=best).", label="tab:nemenyi")
utils.save_table(nem["pairwise"], "nemenyi_pairwise")

import seaborn as sns
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=ranks.values, y=ranks.index, ax=ax, hue=ranks.index, legend=False)
ax.axvline(ranks.min() + nem["critical_difference"], ls="--", color="red",
           label=f"CD = {nem['critical_difference']:.2f}")
ax.set_title("Average ranks across CV folds (lower = better)")
ax.set_xlabel("Average rank"); ax.legend()
fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "nemenyi_ranks.png", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved nemenyi_ranks.png")

Critical difference (alpha=0.05): 9.5912


,avg_rank
Random Forest,2.0
Extra Trees,2.6
CatBoost,3.2
XGBoost,4.2
Gradient Boosting,4.7
LightGBM,5.6
Decision Tree,6.8
MLP,7.5
KNN,8.4
SVM,10.2


Saved nemenyi_ranks.png


**Interpretation.** If Friedman rejects the null, the models are not all equivalent. The Nemenyi CD then tells us which gaps are meaningful: models whose average-rank difference is below the CD are statistically indistinguishable. In practice the top ensembles cluster together (no significant difference among them), while all of them differ significantly from the linear baselines.